# 🗺️ Fas 1c: Geokodning — Avstånd till centrum, skolor & stationer

**Syfte:** Beräkna koordinater för varje bostad och avstånd till viktiga platser  
**Metod:** Geokoda områden via OpenStreetMap (Nominatim) — gratis, ingen API-nyckel  
**Tid:** ~5 minuter (vi geokodar 40 områden, inte 6600 adresser)

In [ ]:
# ============================================================
# STEG 1: Installera och importera
# ============================================================
import subprocess
subprocess.run(['pip', 'install', 'geopy'], capture_output=True)

import pandas as pd
import numpy as np
import re
import time
import json
import os
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
from geopy.extra.rate_limiter import RateLimiter

print('Paket laddade!')

In [ ]:
# ============================================================
# STEG 2: Ladda data och extrahera adresser
# ============================================================
df = pd.read_csv('../data/processed/orebro_housing_enriched.csv')
print(f'Laddad: {len(df)} rader')

# Extrahera gatuadress från raw_text
# raw_text börjar med: "Såld 13 mar. 2026 Kaserngården 14 Lägenhet Rynninge..."
def extract_address(raw_text):
    if pd.isna(raw_text):
        return None
    # Mönster: efter datumet, före "Lägenhet/Villa/Radhus"
    match = re.search(
        r'(?:Såld.*?\d{4})\s+(.+?)\s+(?:Lägenhet|Villa|Radhus|Bostadsrätt|Fritidshus)',
        str(raw_text)
    )
    if match:
        return match.group(1).strip()
    return None

if 'raw_text' in df.columns:
    df['gatuadress'] = df['raw_text'].apply(extract_address)
    print(f'Extraherade adresser: {df["gatuadress"].notna().sum()} av {len(df)}')
    print(f'\nExempel:')
    for addr in df['gatuadress'].dropna().head(5):
        print(f'  {addr}')
else:
    print('Ingen raw_text-kolumn — vi använder omrade_clean istället')

In [ ]:
# ============================================================
# STEG 3: Definiera viktiga platser i Örebro
# ============================================================

# Örebro Stortorget (centrum)
CENTRUM = (59.2753, 15.2134)

# Örebro centralstation
CENTRALSTATION = (59.2773, 15.2066)

# Universitetssjukhuset Örebro (USÖ)
SJUKHUSET = (59.2722, 15.1758)

# Örebro universitet
UNIVERSITETET = (59.2537, 15.2472)

# Marieberg köpcentrum
MARIEBERG = (59.2878, 15.2517)

print('Viktiga platser definierade:')
platser = {
    'centrum': CENTRUM,
    'centralstation': CENTRALSTATION,
    'sjukhuset': SJUKHUSET,
    'universitetet': UNIVERSITETET,
    'marieberg': MARIEBERG,
}
for namn, coord in platser.items():
    print(f'  {namn}: {coord}')

In [ ]:
# ============================================================
# STEG 4: Geokoda alla unika områden
# ============================================================
#
# Istället för att geokoda 6600 adresser (tar timmar),
# geokodar vi ~60 unika områden (tar minuter).
# Varje bostad i samma område får samma koordinat.

geolocator = Nominatim(user_agent='orebro_housing_ml_project')
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)

# Hämta unika områden
omraden = df['omrade_clean'].dropna().unique().tolist()
print(f'Unika områden att geokoda: {len(omraden)}')

# Geokoda varje område
area_coords = {}
failed = []

for omrade in omraden:
    query = f'{omrade}, Örebro kommun, Sverige'
    try:
        location = geocode(query)
        if location:
            area_coords[omrade] = (location.latitude, location.longitude)
        else:
            # Försök med bara områdesnamnet + Örebro
            location = geocode(f'{omrade}, Örebro, Sverige')
            if location:
                area_coords[omrade] = (location.latitude, location.longitude)
            else:
                failed.append(omrade)
    except Exception as e:
        failed.append(omrade)

print(f'\nGeokodade: {len(area_coords)} av {len(omraden)}')
print(f'Misslyckade: {len(failed)}')
if failed:
    print(f'  {failed}')

In [ ]:
# ============================================================
# STEG 5: Fixa misslyckade områden manuellt
# ============================================================
#
# Områden som Nominatim inte hittade sätts till centrum som fallback,
# eller manuella koordinater om vi vet var de ligger.

# Manuella koordinater för vanliga Örebro-områden som kan misslyckas
manuella = {
    'Örebro': CENTRUM,
    'Centralt': CENTRUM,
    'Centralt Öster': (59.2753, 15.2200),
    'Centralt Väster': (59.2753, 15.2050),
    'Öster': (59.2753, 15.2300),
    'Väster': (59.2753, 15.1950),
    'Söder': (59.2650, 15.2134),
    'Norr': (59.2850, 15.2134),
}

for omrade in failed:
    if omrade in manuella:
        area_coords[omrade] = manuella[omrade]
    else:
        # Fallback till centrum
        area_coords[omrade] = CENTRUM
        print(f'  {omrade} → fallback till centrum')

print(f'Totalt geokodade: {len(area_coords)}')

# Visa alla områden med koordinater
print(f'\nExempel:')
for omrade, coord in list(area_coords.items())[:10]:
    dist = geodesic(coord, CENTRUM).km
    print(f'  {omrade:25s}: ({coord[0]:.4f}, {coord[1]:.4f}) — {dist:.1f} km från centrum')

In [ ]:
# ============================================================
# STEG 6: Beräkna avstånd för varje bostad
# ============================================================

# Koppla koordinater till varje bostad via omrade_clean
df['latitude'] = df['omrade_clean'].map(lambda x: area_coords.get(x, CENTRUM)[0])
df['longitude'] = df['omrade_clean'].map(lambda x: area_coords.get(x, CENTRUM)[1])

# Beräkna avstånd till viktiga platser (i km)
def calc_distance(row, target):
    try:
        return geodesic((row['latitude'], row['longitude']), target).km
    except:
        return np.nan

df['avstand_centrum_km'] = df.apply(lambda r: calc_distance(r, CENTRUM), axis=1)
df['avstand_station_km'] = df.apply(lambda r: calc_distance(r, CENTRALSTATION), axis=1)
df['avstand_sjukhus_km'] = df.apply(lambda r: calc_distance(r, SJUKHUSET), axis=1)
df['avstand_universitet_km'] = df.apply(lambda r: calc_distance(r, UNIVERSITETET), axis=1)
df['avstand_marieberg_km'] = df.apply(lambda r: calc_distance(r, MARIEBERG), axis=1)

print('Avstånd beräknade!')
print(f'\navstand_centrum_km:')
print(f'  Min:    {df["avstand_centrum_km"].min():.1f} km')
print(f'  Median: {df["avstand_centrum_km"].median():.1f} km')
print(f'  Max:    {df["avstand_centrum_km"].max():.1f} km')

print(f'\nNärmast centrum: {df.loc[df["avstand_centrum_km"].idxmin(), "omrade_clean"]}')
print(f'Längst från centrum: {df.loc[df["avstand_centrum_km"].idxmax(), "omrade_clean"]}')

In [ ]:
# ============================================================
# STEG 7: Korrelationscheck — påverkar avstånd priset?
# ============================================================

avstand_cols = ['avstand_centrum_km', 'avstand_station_km', 
                'avstand_sjukhus_km', 'avstand_universitet_km',
                'avstand_marieberg_km']

print('Korrelation med slutpris:')
print('=' * 55)
for col in avstand_cols:
    c = df[[col, 'slutpris']].dropna().corr().iloc[0, 1]
    bar = '█' * int(abs(c) * 40)
    sign = '+' if c > 0 else '-'
    print(f'  {col:30s}: {sign}{abs(c):.3f}  {bar}')

# Också latitude och longitude
for col in ['latitude', 'longitude']:
    c = df[[col, 'slutpris']].dropna().corr().iloc[0, 1]
    print(f'  {col:30s}: {"+" if c > 0 else "-"}{abs(c):.3f}')

In [ ]:
# ============================================================
# STEG 8: Spara den berikade datan med koordinater
# ============================================================

output_path = '../data/processed/orebro_housing_enriched.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'Sparad: {output_path}')
print(f'Rader: {len(df)}')
print(f'Kolumner: {len(df.columns)}')

print(f'\nNya kolumner:')
new = ['latitude', 'longitude', 'gatuadress',
       'avstand_centrum_km', 'avstand_station_km', 
       'avstand_sjukhus_km', 'avstand_universitet_km',
       'avstand_marieberg_km']
for col in new:
    if col in df.columns:
        print(f'  + {col}')

# Spara även koordinater separat (för kartan)
coords_df = pd.DataFrame([
    {'omrade': k, 'lat': v[0], 'lon': v[1]} 
    for k, v in area_coords.items()
])
coords_df.to_csv('../data/processed/area_coordinates.csv', index=False)
print(f'\nKoordinater sparade: {len(coords_df)} områden')

print(f'\n{"="*50}')
print(f'KLART!')
print(f'Lägg till avstand_centrum_km i modellens features')
print(f'Koordinaterna används sedan för kartan i dashboarden')
print(f'{"="*50}')